# Banking & Fintech Consumer Complaint Analysis

This notebook loads the sample dataset and reproduces the charts saved in `outputs/`.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

DATA_PATH = 'data/consumer_complaints_banking_sample_20000.csv'
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'Missing {DATA_PATH}. Upload the sample CSV into data/.')

df = pd.read_csv(DATA_PATH)
df['Date received'] = pd.to_datetime(df['Date received'], errors='coerce')
df['Month'] = df['Date received'].dt.to_period('M').astype(str)

df.head()


In [ ]:
plt.rcParams.update({'figure.dpi':200, 'savefig.dpi':300, 'font.size':10, 'axes.titlesize':12})

def polish(ax):
    ax.grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.6)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def save_bar(series, title, xlabel, ylabel, filename, top_n=10):
    s = series.dropna().astype(str).str.strip().value_counts().head(top_n).sort_values()
    fig, ax = plt.subplots(figsize=(10,5))
    ax.barh(s.index, s.values)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    polish(ax)
    fig.tight_layout()
    out_path = os.path.join('outputs', filename)
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)
    print('Saved:', out_path)

save_bar(df['Product'], 'Top Products by Complaint Volume (Banking/Fintech)', 'Complaints', 'Product', '01_top_products.png', top_n=9)
save_bar(df['Issue'], 'Top Issues by Complaint Volume', 'Complaints', 'Issue', '02_top_issues.png', top_n=10)
save_bar(df['Company response to consumer'], 'Top Company Response Types', 'Count', 'Response type', '03_response_types.png', top_n=10)
save_bar(df['State'], 'Top States by Complaint Volume', 'Complaints', 'State', '04_top_states.png', top_n=10)


In [ ]:
trend = df.dropna(subset=['Month']).groupby('Month').size()
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(trend.index, trend.values, marker='o', linewidth=1.5)
ax.set_title('Complaints Over Time (Monthly)')
ax.set_xlabel('Month')
ax.set_ylabel('Complaints')
ax.tick_params(axis='x', rotation=45)
polish(ax)
fig.tight_layout()
out_path = os.path.join('outputs', '05_trend_monthly.png')
fig.savefig(out_path, bbox_inches='tight')
plt.close(fig)
print('Saved:', out_path)
